In [ ]:
import csv
import os
import uuid
import tqdm
import geopandas as gpd
import numpy as np
from PIL import Image
import rasterio
from rasterio.crs import CRS
from rasterio.warp import transform_bounds
from shapely.geometry import box


def tile_raster_by_landmass(
    raster_path: str,
    shapefile_path: str,
    output_dir: str,
    csv_out_path: str,
    tile_size: int = 224,
):
    """Tiles a GeoTIFF into tile_size x tile_size PNGs if they overlap with a landmass shapefile.

    Saves a CSV containing unique IDs, filenames, and centroid Lat/Lon (EPSG:4326).
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1. Load Landmass Shapefile
    print("Loading landmass shapefile...")
    land_gdf = gpd.read_file(shapefile_path)

    # Ensure shapefile is valid and has spatial index
    land_gdf = land_gdf[land_gdf.geometry.notnull()]
    land_sindex = land_gdf.sindex

    # Open GeoTIFF
    with rasterio.open(raster_path) as src:
        width = src.width
        height = src.height
        raster_crs = src.crs
        transform = src.transform

        print(
            f"Raster Size: {width}x{height} | Bands: {src.count} | CRS: {raster_crs}"
        )

        # Ensure Land Geometry CRS matches Raster CRS for fast bounding-box checks
        if land_gdf.crs != raster_crs:
            print(
                f"Reprojecting landmass geometry from {land_gdf.crs} to {raster_crs}..."
            )
            land_gdf_reprojected = land_gdf.to_crs(raster_crs)
        else:
            land_gdf_reprojected = land_gdf

        # CSV Preparation
        records = []

        # Target CRS for CSV output (Lat/Lon = EPSG:4326)
        wgs84_crs = CRS.from_epsg(4326)

        # 2. Iterate through raster grid in tile_size steps
        total_tiles_processed = 0
        land_tiles_saved = 0

        print("Starting tiling loop...")

        for y in tqdm.tqdm(range(0, height - tile_size + 1, tile_size)):
            for x in range(0, width - tile_size + 1, tile_size):
                total_tiles_processed += 1

                # Pixel window coordinates to Geographic/Projected Bounding Box
                # Bounds: (minx, miny, maxx, maxy)
                tile_win_bounds = rasterio.windows.bounds(
                    rasterio.windows.Window(x, y, tile_size, tile_size),
                    transform,
                )
                tile_box = box(*tile_win_bounds)

                # 3. Spatial Intersect Check against Landmass Shapefile using Spatial Index
                possible_matches_index = list(
                    land_sindex.intersection(tile_box.bounds)
                )
                possible_matches = land_gdf_reprojected.iloc[
                    possible_matches_index
                ]

                # Check actual intersection
                if not possible_matches.intersects(tile_box).any():
                    continue  # Skip tile - pure ocean / outside landmass

                # 4. Read RGB Image Data for Tile Window
                window = rasterio.windows.Window(x, y, tile_size, tile_size)
                # Read 3 bands (RGB)
                rgb_data = src.read([1, 2, 3], window=window)

                # Transpose from (Bands, Height, Width) to (Height, Width, Bands) for PIL Image
                rgb_array = np.transpose(rgb_data, (1, 2, 0)).astype(np.uint8)

                # Skip completely black/nodata filled tiles if necessary
                if not np.any(rgb_array):
                    continue

                # 5. Calculate Centroid (Lat / Lon in EPSG:4326)
                centroid = tile_box.centroid
                if raster_crs != wgs84_crs:
                    # Reproject centroid to WGS84 (Lat/Lon)
                    cx_bounds = transform_bounds(
                        raster_crs,
                        wgs84_crs,
                        centroid.x,
                        centroid.y,
                        centroid.x,
                        centroid.y,
                    )
                    lon, lat = cx_bounds[0], cx_bounds[1]
                else:
                    lon, lat = centroid.x, centroid.y

                # 6. Generate Unique ID & File Output
                tile_id = str(uuid.uuid4())
                filename = f"{tile_id}.png"
                out_png_path = os.path.join(output_dir, filename)

                # Save Image using PIL
                img = Image.fromarray(rgb_array)
                img.save(out_png_path, format="PNG")

                # Store metadata record
                records.append(
                    {
                        "tile_id": tile_id,
                        "filename": filename,
                        "lat": round(lat, 6),
                        "lon": round(lon, 6),
                        "col_off": x,
                        "row_off": y,
                    }
                )

                land_tiles_saved += 1

                if land_tiles_saved % 1000 == 0:
                    print(f"Saved {land_tiles_saved} land tiles...")

        # 7. Write Index CSV
        print(f"Saving metadata CSV to {csv_out_path}...")
        with open(csv_out_path, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(
                f,
                fieldnames=[
                    "tile_id",
                    "filename",
                    "lat",
                    "lon",
                    "col_off",
                    "row_off",
                ],
            )
            writer.writeheader()
            writer.writerows(records)

        print(f"\nProcessing Complete!")
        print(f"Total Grid Windows Tested: {total_tiles_processed}")
        print(f"Total Land Tiles Saved: {land_tiles_saved}")


if __name__ == "__main__":
    # Define File Paths
    INPUT_GEOTIFF = rf"E:\Data\satclip\world_rgb\blue_marble_global_stitched.tif"
    LAND_SHAPEFILE = rf"E:\Data\Global\World\land-poly\land_polygons.shp"
    OUTPUT_FOLDER = RF"E:\Data\satclip\world_rgb\tiles"
    OUTPUT_CSV = RF"E:\Data\satclip\world_rgb\tiles_META.CSV"

    tile_raster_by_landmass(
        raster_path=INPUT_GEOTIFF,
        shapefile_path=LAND_SHAPEFILE,
        output_dir=OUTPUT_FOLDER,
        csv_out_path=OUTPUT_CSV,
        tile_size=224,
    )

In [ ]:
import math
import os
import pandas as pd
from PIL import Image
from scipy.special import sph_harm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ==========================================
# 1. SPHERICAL HARMONICS POSITION ENCODING (L=40)
# ==========================================

class SphericalHarmonicsPE(nn.Module):
    """
    Computes real spherical harmonics up to degree L_max for input (lat, lon) coordinates.
    For degree L, the number of harmonic components is (L_max + 1)^2.
    For L=40, the output embedding dimension is (40 + 1)^2 = 1681.
    """
    def __init__(self, l_max: int = 40):
        super().__init__()
        self.l_max = l_max
        self.out_dim = (l_max + 1) ** 2

    def _latlon_to_spherical(self, lat_deg: torch.Tensor, lon_deg: torch.Tensor):
        # Convert lat/lon in degrees to spherical coordinates (theta, phi) in radians
        # theta (colatitude) in [0, pi], phi (azimuth) in [0, 2*pi]
        lat_rad = torch.deg2rad(lat_deg)
        lon_rad = torch.deg2rad(lon_deg)
        
        theta = torch.pi / 2.0 - lat_rad  # colatitude
        phi = torch.remainder(lon_rad + 2 * torch.pi, 2 * torch.pi)  # azimuth
        return theta, phi

    def forward(self, lat: torch.Tensor, lon: torch.Tensor) -> torch.Tensor:
        """
        Args:
            lat: Tensor of shape (N,)
            lon: Tensor of shape (N,)
        Returns:
            Tensor of shape (N, (l_max + 1)^2)
        """
        theta, phi = self._latlon_to_spherical(lat, lon)
        
        theta_np = theta.detach().cpu().numpy()
        phi_np = phi.detach().cpu().numpy()
        
        sh_components = []
        for l in range(self.l_max + 1):
            for m in range(-l, l + 1):
                # Calculate complex spherical harmonic Y_l^m(phi, theta)
                # Note: scipy.special.sph_harm signature is sph_harm(m, n, theta, phi)
                y_lm = sph_harm(abs(m), l, phi_np, theta_np)
                
                # Convert complex spherical harmonics to Real Spherical Harmonics
                if m < 0:
                    r_sh = math.sqrt(2) * ((-1) ** m) * y_lm.imag
                elif m == 0:
                    r_sh = y_lm.real
                else:
                    r_sh = math.sqrt(2) * ((-1) ** m) * y_lm.real
                    
                sh_components.append(r_sh)
                
        # Stack all (L+1)^2 components along last axis
        sh_matrix = torch.from_numpy(
            torch.tensor(sh_components, dtype=torch.float32).T.values 
            if hasattr(sh_components[0], 'values') 
            else torch.tensor(sh_components, dtype=torch.float32).T
        ).to(lat.device)
        
        return sh_matrix


# ==========================================
# 2. DATASET & PATCH-LEVEL CENTROID CALCULATOR
# ==========================================

class TilePatchDataset(Dataset):
    """
    Loads 224x224 RGB PNG tiles and computes exact patch-level lat/lon coordinates.
    """
    def __init__(self, csv_file: str, tiles_dir: str, patch_size: int = 14, img_size: int = 224):
        self.df = pd.read_csv(csv_file)
        self.tiles_dir = tiles_dir
        self.patch_size = patch_size
        self.img_size = img_size
        self.num_patches_per_side = img_size // patch_size
        
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.tiles_dir, row["filename"])
        
        image = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(image)
        
        # Tile centroid coordinates & spatial attributes
        tile_lat = row["lat"]
        tile_lon = row["lon"]
        
        # Compute Lat/Lon grid for each patch token within the 224x224 tile
        # Assuming Web Mercator/Equirectangular local projection scale across 224px
        # Approximate meters/degree delta offset per pixel for local patch displacement
        patch_lats = []
        patch_lons = []
        
        # Grid step relative to center of tile
        half_grid = self.num_patches_per_side / 2.0
        
        # Small delta offset per patch across 224px image window
        lat_step = 0.001  # local latitude delta step per patch
        lon_step = 0.001 / max(math.cos(math.radians(tile_lat)), 1e-5)
        
        for i in range(self.num_patches_per_side):      # Row (y)
            for j in range(self.num_patches_per_side):  # Col (x)
                # Offset relative to tile centroid
                py = (i + 0.5) - half_grid
                px = (j + 0.5) - half_grid
                
                p_lat = tile_lat - (py * lat_step)
                p_lon = tile_lon + (px * lon_step)
                
                patch_lats.append(p_lat)
                patch_lons.append(p_lon)
                
        patch_lats = torch.tensor(patch_lats, dtype=torch.float32)
        patch_lons = torch.tensor(patch_lons, dtype=torch.float32)
        
        return img_tensor, patch_lats, patch_lons


# ==========================================
# 3. JEPA ARCHITECTURE (DINO BACKBONE + SH PRE DICTOR)
# ==========================================

class LocationConditionedJEPA(nn.Module):
    def __init__(self, sh_lmax: int = 40, patch_embed_dim: int = 768, hidden_dim: int = 1024):
        super().__init__()
        
        # 1. DINOv2 ViT Backbone (Fixed/Frozen Target Encoder)
        # Yields (B, 256, 768) embeddings for 14x14 patches on 224x224 input
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        for param in self.backbone.parameters():
            param.requires_grad = False  # Freeze Target Teacher Encoder
            
        self.sh_encoder = SphericalHarmonicsPE(l_max=sh_lmax)
        sh_dim = self.sh_encoder.out_dim  # (40 + 1)^2 = 1681
        
        # 2. JEPA Predictor Model
        # Predicts DINO patch embeddings conditioned ON Spherical Harmonics position embeddings
        self.predictor = nn.Sequential(
            nn.Linear(sh_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, patch_embed_dim)
        )

    def forward(self, images: torch.Tensor, patch_lats: torch.Tensor, patch_lons: torch.Tensor):
        B, N_patches = patch_lats.shape[0], patch_lats.shape[1]
        
        # 1. Target Latent Representation from DINO Backbone (No gradients)
        with torch.no_grad():
            # Get ViT patch tokens excluding [CLS] token
            features = self.backbone.forward_features(images)
            target_patch_embs = features["x_norm_patchtokens"]  # Shape: (B, 256, 768)
            
        # 2. Compute Spherical Harmonics (L=40) for all patch centroids
        flat_lats = patch_lats.view(-1)
        flat_lons = patch_lons.view(-1)
        
        sh_embeds = self.sh_encoder(flat_lats, flat_lons)  # Shape: (B * 256, 1681)
        
        # 3. JEPA Predictor Reconstruction
        pred_patch_embs = self.predictor(sh_embeds)  # Shape: (B * 256, 768)
        pred_patch_embs = pred_patch_embs.view(B, N_patches, -1)
        
        return pred_patch_embs, target_patch_embs


# ==========================================
# 4. TRAINING LOOP
# ==========================================

def train_jepa_pipeline(
    csv_file: str,
    tiles_dir: str,
    epochs: int = 10,
    batch_size: int = 16,
    lr: float = 1e-4,
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
):
    print(f"Initializing dataset and model on device: {device}...")
    
    dataset = TilePatchDataset(csv_file=csv_file, tiles_dir=tiles_dir)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    
    model = LocationConditionedJEPA(sh_lmax=40, patch_embed_dim=768).to(device)
    optimizer = torch.optim.AdamW(model.predictor.parameters(), lr=lr, weight_decay=1e-2)
    
    print("Starting JEPA Training Loop...")
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0.0
        
        for step, (images, patch_lats, patch_lons) in enumerate(dataloader):
            images = images.to(device)
            patch_lats = patch_lats.to(device)
            patch_lons = patch_lons.to(device)
            
            optimizer.zero_grad()
            
            # Predict DINO latent representations from Spherical Harmonics
            pred_embs, target_embs = model(images, patch_lats, patch_lons)
            
            # JEPA Loss: Smooth L1 / MSE loss in latent embedding space
            loss = F.smooth_l1_loss(pred_embs, target_embs)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            if step % 1 == 0:
                print(f"Epoch [{epoch+1}/{epochs}] | Step [{step}/{len(dataloader)}] | JEPA Latent Loss: {loss.item():.6f}")
                
        avg_loss = total_loss / len(dataloader)
        print(f"--> Epoch [{epoch+1}/{epochs}] Finished | Average Loss: {avg_loss:.6f}\n")
        
    # Save trained JEPA Predictor weights
    torch.save(model.state_dict(), "jepa_sh40_reconstructor.pth")
    print("Training Complete! Model weights saved to 'jepa_sh40_reconstructor.pth'.")


if __name__ == "__main__":
    TILES_DIR = RF"E:\Data\satclip\world_rgb\tiles"
    CSV_FILE = RF"E:\Data\satclip\world_rgb\tiles_META.CSV"
    
    train_jepa_pipeline(
        csv_file=CSV_FILE,
        tiles_dir=TILES_DIR,
        epochs=5,
        batch_size=1024,
        lr=3e-4
    )

Initializing dataset and model on device: cuda...


Using cache found in C:\Users\sus14836/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\sus14836/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\sus14836/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\sus14836/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Starting JEPA Training Loop...


In [11]:
import sys

sys.path.append(rf"D:\Code\satclip")

In [12]:
from exp import infer

In [13]:
infer(25, 125)

Using cache found in C:\Users\sus14836/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\sus14836/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\sus14836/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\sus14836/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


(tensor([[-0.0086,  2.9068,  0.2036, -1.6450, -0.3256,  1.2570, -1.6634,  0.7760,
          -0.4995,  0.2642, -1.0065, -0.7730, -1.7740,  0.4750,  0.4481, -0.6396,
           0.5529,  0.7044, -1.1330, -0.3152,  0.5095,  0.0934,  0.0643,  0.1796,
           1.6465,  0.3466, -0.3710,  0.1221,  0.9621,  0.3918, -0.0269, -0.1630,
          -0.4058, -0.7637, -1.0396, -0.1588,  1.2079,  0.2280, -1.8676, -0.2594,
          -0.3398,  1.7004,  1.4899, -1.9069, -0.7765, -1.3944, -1.3001,  1.0321,
           0.0704,  1.0576, -0.3758, -1.4732, -1.4445, -0.5850,  0.2575, -0.0737,
           1.4165, -0.1592,  1.6350, -0.4318,  1.4026,  0.9255,  0.0584,  0.7139]],
        device='cuda:0'),
 tensor([[-1.8122e-02, -1.2389e-01,  5.6529e-01, -2.3022e-01,  4.2476e-01,
           6.2609e-01,  3.0607e-02, -3.9293e-02, -7.3142e-01,  5.3800e-01,
          -4.9882e-01,  7.3809e-01, -4.1052e-01, -1.4870e-01, -7.0132e-01,
           2.7690e-02,  1.1317e+00,  3.4619e-01, -1.9804e-01, -9.2148e-01,
          -4.303